In [1]:
!pip install gradio tensorflow scikit-learn pandas matplotlib

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM,Dense,Dropout
import gradio as gr
import datetime

In [3]:
def prepare_data(file_path):
  df=pd.read_csv(file_path)
  df['date']=pd.to_datetime(df['date'])
  df.set_index('date',inplace=True)

  daily_df=df.resample('D').agg({
      'open':'first',
      'high':'max',
      'low':'min',
      'close':'last',
      'volume':'sum'
  }).dropna()

  return daily_df

In [6]:
def build_and_train_model(data):
  dataset=data[['close']].values
  scaler=MinMaxScaler(feature_range=(0,1))
  scaled_data=scaler.fit_transform(dataset)

  prediction_days=60
  x_train,y_train=[],[]

  for x in range(prediction_days,len(scaled_data)):
    x_train.append(scaled_data[x-prediction_days:x,0])
    y_train.append(scaled_data[x,0])

    x_train,y_train=np.array(x_train),np.array(y_train)
    x_train=np.reshape(x_train,(x_train.shape[0],x_train.shape[1],1))
    model=Sequential([
      LSTM(units=50,return_sequences=True,input_shape=(x_train.shape[1],1)),
      Dropout(0.2),
      LSTM(units=50,return_sequences=False),
      Dropout(0.2),
      Dense(units=25),
      Dense(units=1)
    ])

    model.compile(optimizer='adam',loss='mean_squared_error')
    model.fit(x_train,y_train,epochs=5,batch_size=32,verbose=0)

    return model,scaler,dataset,prediction_days
print("Preprocessing data and training model....Please wait.")
daily_data=prepare_data('/content/TATASTEEL_minute.csv')
model,scaler,raw_dataset,lookback=build_and_train_model(daily_data)
print("Model trained successfully!")

Preprocessing data and training model....Please wait.


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model trained successfully!


In [10]:
def predict_next_day():
  real_data=[raw_dataset[len(raw_dataset)-lookback:len(raw_dataset),0]]
  real_data=np.array(real_data)
  real_data=np.reshape(real_data,(real_data.shape[0],real_data.shape[1],1))
  real_data_scaled=scaler.transform(real_data.reshape(-1,1)).reshape(1,lookback,1)

  prediction=model.predict(real_data_scaled)
  predicted_price=scaler.inverse_transform(prediction)[0][0]

  last_price=raw_dataset[-1][0]
  trend="Bullish(Up)" if predicted_price >last_price else "Bearish(Down)"
  change=((predicted_price - last_price)/last_price)*100

  plt.figure(figsize=(10,4))
  plt.plot(daily_data.index[-30:],daily_data['close'][-30:],label='Actual(Last 30 Days)')
  plt.axhline(y=predicted_price,color='r',linestyle='--',label=f'Predicted:{predicted_price:.2f}')
  plt.title('Tata Steel Price Trend Prediction')
  plt.legend()
  plt.grid(True)
  plt.savefig('prediction_plot.png')
  plt.close()

  return f"₹{predicted_price:.2f}",trend,f"{change:.2f}%",'prediction_plot.png'

In [ ]:
with gr.Blocks(theme=gr.themes.Soft()) as demo:
  gr.Markdown("# Tata Steel Market Predictor")
  gr.Markdown("This AI Model uses an LSTM network to analyse historical minute-level data and predict the next day's closing price.")

  with gr.Row():
    with gr.Column():
      predict_btn=gr.Button("Predict Next Day Trend", variant="primary")
      last_date=daily_data.index[-1].strftime('%Y-%m-%d')
      gr.Markdown(f"Model trained on data up to : {last_date}") # Changed gr.info to gr.Markdown

    with gr.Column():
      out_price=gr.Textbox(label="Predicted Closing Price")
      out_trend=gr.Textbox(label="Market Trend")
      out_change=gr.Textbox(label="Expected Change(%)")

  out_plot=gr.Image(label="Price Trend Visualisation")
  predict_btn.click(
      fn=predict_next_day,
      outputs=[out_price,out_trend,out_change,out_plot]
  )
demo.launch(debug=True)

/tmp/ipython-input-1226111472.py:1: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://1be4e8e15a309c6f89.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
